# Tutorial 2: Melt-and-Quench of Crystalline SiO₂

This tutorial demonstrates the full **7-stage melt-and-quench pipeline** for
generating an amorphous SiO₂ structure from a crystalline input using AmorphGen.

We use a quick protocol suitable for demonstration:
- **Crystalline input:** α-quartz SiO₂ (2×2×2 supercell, 72 atoms)
- **Backend:** CHGNet (fast on CPU; any backend can be substituted)
- **Heating/cooling rate:** ~2000 K/ps (fast, for tutorial speed)
- **Timestep:** 2 fs (allows fewer steps for same simulation time)

**7-stage pipeline:**
1. Structure optimisation (crystalline)
2. Pre-melt equilibration (NPT, 300 K, 4 ps)
3. Melt — heat ramp (NPT, 300 → 2000 K)
4. High-T equilibration (NPT, 2000 K, 4 ps)
5. Quench — cooling ramp (NPT, 2000 → 300 K)
6. Low-T equilibration (NPT, 300 K, 4 ps)
7. Final optimisation (amorphous)

**Estimated runtime:** ~10–15 min on CPU, ~5 min on GPU

In [1]:
# Install if needed (uncomment)
# !pip install -e "..[mace]"
# !pip install matplotlib

import os
import numpy as np
import matplotlib.pyplot as plt
from ase.io import read, write
from ase.data import atomic_masses, atomic_numbers

In [2]:
#!pip install chgnet

---
## 2.1 Prepare the crystalline input

Load α-quartz SiO₂ and build a 2×2×2 supercell (72 atoms).

In [3]:
# Load crystalline SiO2
atoms = read("SiO2_con.cif")
print(f"Unit cell: {atoms.get_chemical_formula()} ({len(atoms)} atoms)")

# Build 2x2x2 supercell
supercell = atoms * (2, 2, 2)
print(f"Supercell: {supercell.get_chemical_formula()} ({len(supercell)} atoms)")
print(f"Cell: {supercell.cell[0,0]:.3f} x {supercell.cell[1,1]:.3f} x {supercell.cell[2,2]:.3f} Å")

# Compute density
total_mass = sum(atomic_masses[atomic_numbers[s]] for s in supercell.get_chemical_symbols())
vol_cm3 = supercell.get_volume() * 1e-24
density = (total_mass / 6.022e23) / vol_cm3
print(f"Density: {density:.3f} g/cm³ (crystalline α-quartz: 2.65 g/cm³)")

# Save as input for the pipeline
write("SiO2_supercell.extxyz", supercell, format="extxyz")
print("\nSaved -> SiO2_supercell.extxyz")

Unit cell: O6Si3 (9 atoms)
Supercell: O48Si24 (72 atoms)
Cell: 9.830 x 8.513 x 10.863 Å
Density: 2.634 g/cm³ (crystalline α-quartz: 2.65 g/cm³)

Saved -> SiO2_supercell.extxyz


---
## 2.2 Load the MACE calculator

We use chgnet as the force engine. You can substitute `"MACE-MPA-0"` or
`"m3gnet"` here — the rest of the tutorial works identically.

In [9]:
from amorphgen.utils import get_calculator

# CHGNet — fast on CPU, also supports Apple MPS GPU
# Options: "cpu", "cuda" (NVIDIA GPU), "mps" (Apple Silicon GPU)
#  device = "cpu"

# Uncomment to try Apple Metal GPU (may not work for all operations):
device = "mps"

calc = get_calculator(model="chgnet", device=device)
print(f"CHGNet calculator loaded (device: {device}).")

[CHGNet] Loading pretrained model on mps
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on mps
CHGNet will run on mps
CHGNet calculator loaded (device: mps).


---
## 2.3 Run the 7-stage melt-and-quench pipeline

We configure a **quick protocol** for this tutorial:
- NPT ensemble for all MD stages (allows density to adjust)
- ~2000 K/ps heating and cooling rates
- 4 ps equilibration at each temperature
- 2 fs timestep (reduces total steps)
- Melt temperature: 2000 K

For production runs, you would use slower cooling rates (10–100 K/ps),
1 fs timestep, and longer equilibration times (50–100 ps).

In [10]:
from amorphgen import MeltQuenchPipeline

pipe = MeltQuenchPipeline(
    input_file="SiO2_supercell.extxyz",
    work_dir="SiO2_melt_quench",
    cfg_override={
        "model": "chgnet",
        "device": device,

        # Stage 1 & 7: optimisation
        "opt": {
            "fmax": 0.1,                # eV/Å — relaxed for tutorial speed
            "max_steps": 100,
        },

        # Stage 2: pre-melt equilibration (NPT, 300 K, 4 ps)
        "eq_premelt": {
            "ensemble": "NPT",
            "T": 300,
            "steps": 2000,              # 4 ps at 2 fs timestep
            "timestep": 2.0,
            "friction": 0.01,
            "ttime": 25.0,
        },

        # Stage 3: melt — heat ramp (NPT, 300 → 2000 K, ~2000 K/ps)
        "melt": {
            "ensemble": "NPT",
            "T_start": 300,
            "T_end": 2000,
            "T_step": 200,              # 200 K per segment
            "steps_per_T": 50,          # 0.1 ps per segment → ~2000 K/ps
            "timestep": 2.0,
            "friction": 0.01,
            "ttime": 25.0,
        },

        # Stage 4: high-T equilibration (NPT, 2000 K, 4 ps)
        "eq_high": {
            "ensemble": "NPT",
            "T": 2000,
            "steps": 2000,              # 4 ps
            "timestep": 2.0,
            "friction": 0.01,
            "ttime": 25.0,
        },

        # Stage 5: quench — cooling ramp (NPT, 2000 → 300 K, ~2000 K/ps)
        "quench": {
            "ensemble": "NPT",
            "T_start": 2000,
            "T_end": 300,
            "T_step": -200,             # -200 K per segment (cooling)
            "steps_per_T": 50,          # 0.1 ps per segment → ~2000 K/ps
            "timestep": 2.0,
            "friction": 0.01,
            "ttime": 25.0,
        },

        # Stage 6: low-T equilibration (NPT, 300 K, 4 ps)
        "eq_low": {
            "ensemble": "NPT",
            "T": 300,
            "steps": 2000,              # 4 ps
            "timestep": 2.0,
            "friction": 0.01,
            "ttime": 25.0,
        },
    },
)

# Run all 7 stages
final_atoms = pipe.run()
print(f"\nFinal structure: {len(final_atoms)} atoms")

[CHGNet] Loading pretrained model on mps
CHGNet v0.3.0 initialized with 412,525 parameters
CHGNet will run on mps
CHGNet will run on mps

═════════════════════════════════════════════════════════════════
  AmorphGen  v2.0  —  Melt-and-Quench Pipeline
  Model:  chgnet
  Input:  SiO2_supercell.extxyz
  Stages: [1, 2, 3, 4, 5, 6, 7]
  Output: SiO2_melt_quench/
═════════════════════════════════════════════════════════════════


─────────────────────────────────────────────────────────────────
  Stage 1: Structure optimisation (crystalline)
─────────────────────────────────────────────────────────────────

[Opt] Using provided Atoms object

  Composition: O48Si24 (72 atoms)
  Initial cell: a=9.8299  b=9.8299  c=10.8626
  Volume: 909.00 A^3
  Optimizer: LBFGS  fmax=0.1  max_steps=100

   Step      Energy(eV)   Fmax(eV/A)        a(A)        b(A)        c(A)     Vol(A3)
  -------------------------------------------------------------------------------------


/Users/c.kaewmeechai@bham.ac.uk/Library/jupyterlab-desktop/jlab_server/lib/python3.12/site-packages/chgnet/model/model.py:898: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  volumes = torch.tensor(volumes, dtype=TORCH_DTYPE, device=atomic_numbers.device)


      1     -599.881165     0.383789    9.830733    9.830733   10.863449    909.2225
      2     -600.016174     0.210408    9.835969    9.835969   10.868907    910.6487
      3     -600.019653     0.206372    9.836470    9.836470   10.869403    910.7829
      4     -600.074280     0.114566    9.852413    9.852413   10.885170    915.0631
      5     -600.076294     0.120851    9.852793    9.852793   10.885506    915.1621
      6     -600.084961     0.139847    9.855890    9.855890   10.888277    915.9705
      7     -600.098755     0.158997    9.862160    9.862161   10.893874    917.6078
      8     -600.142273     0.274777    9.883881    9.883882   10.913327    923.3000
      9     -600.187378     0.360419    9.911186    9.911187   10.937852    930.4948
     10     -600.251221     0.336016    9.938491    9.938492   10.962341    937.7237
     11     -600.312500     0.272721    9.965797    9.965798   10.986751    944.9829
     12     -600.363464     0.232331    9.993102    9.993103   11

### Equivalent CLI command

The same run can be done from the command line:

```bash
amorphgen SiO2_supercell.extxyz \
    --model chgnet \
    --device cpu \
    --work-dir SiO2_melt_quench \
    --fmax 0.1 \
    --opt-steps 100 \
    --eq-premelt-ensemble NPT \
    --eq-premelt-T 300 \
    --eq-premelt-steps 2000 \
    --melt-ensemble NPT \
    --melt-T-start 300 \
    --melt-T-end 2000 \
    --melt-T-step 200 \
    --melt-steps-per-T 50 \
    --eq-high-ensemble NPT \
    --eq-high-T 2000 \
    --eq-high-steps 2000 \
    --quench-ensemble NPT \
    --quench-T-start 2000 \
    --quench-T-end 300 \
    --quench-T-step -200 \
    --quench-steps-per-T 50 \
    --eq-low-ensemble NPT \
    --eq-low-T 300 \
    --eq-low-steps 2000
```

---
## 2.4 Compare crystalline vs amorphous structure

### Density comparison

In [ ]:
def get_density(atoms):
    total_mass = sum(atomic_masses[atomic_numbers[s]] for s in atoms.get_chemical_symbols())
    vol_cm3 = atoms.get_volume() * 1e-24
    return (total_mass / 6.022e23) / vol_cm3

crystal = read("SiO2_supercell.extxyz")
amorphous = read("SiO2_melt_quench/stage7_amorphous_final.cif")

rho_crystal = get_density(crystal)
rho_amorphous = get_density(amorphous)
reduction = (1 - rho_amorphous / rho_crystal) * 100

print(f"{'Structure':<20s}  {'Atoms':>6s}  {'Density (g/cm³)':>16s}")
print("-" * 50)
print(f"{'Crystalline':<20s}  {len(crystal):6d}  {rho_crystal:16.3f}")
print(f"{'Amorphous':<20s}  {len(amorphous):6d}  {rho_amorphous:16.3f}")
print(f"\nDensity reduction: {reduction:.1f}%")
print(f"Experimental amorphous SiO₂: ~2.20 g/cm³ (fused silica)")

### Radial Distribution Function (RDF)

The RDF is the primary structural fingerprint for amorphous materials.
Key features of amorphous SiO₂:
- **Si–O peak** at ~1.62 Å (tetrahedral bond)
- **O–O peak** at ~2.63 Å (edge of SiO₄ tetrahedron)
- **Si–Si peak** at ~3.12 Å (corner-sharing tetrahedra)
- Loss of long-range order beyond ~5 Å

In [ ]:
def compute_rdf(atoms, rmax=8.0, nbins=200):
    """Compute the total radial distribution function."""
    d = atoms.get_all_distances(mic=True)
    np.fill_diagonal(d, np.inf)
    pairs = d[np.triu_indices_from(d, k=1)]
    pairs = pairs[pairs < rmax]
    hist, edges = np.histogram(pairs, bins=nbins, range=(0, rmax))
    r = 0.5 * (edges[:-1] + edges[1:])
    dr = edges[1] - edges[0]
    n = len(atoms)
    vol = atoms.get_volume()
    n_pairs = n * (n - 1) / 2
    shell_vol = 4 * np.pi * r**2 * dr
    g_r = hist * vol / (n_pairs * shell_vol)
    return r, g_r

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Crystalline
r_c, g_c = compute_rdf(crystal)
ax1.plot(r_c, g_c, color="steelblue", lw=1.5)
ax1.set_title("Crystalline α-quartz SiO₂", fontweight="bold")
ax1.set_xlabel("r (Å)")
ax1.set_ylabel("g(r)")
ax1.set_xlim(0, 8)

# Amorphous
r_a, g_a = compute_rdf(amorphous)
ax2.plot(r_a, g_a, color="orangered", lw=1.5)
ax2.set_title("Amorphous SiO₂ (melt-quench)", fontweight="bold")
ax2.set_xlabel("r (Å)")
ax2.set_ylabel("g(r)")
ax2.set_xlim(0, 8)

fig.suptitle("RDF comparison: crystalline vs amorphous SiO₂", fontsize=14)
plt.tight_layout()
plt.savefig("rdf_comparison_SiO2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> rdf_comparison_SiO2.png")

In [ ]:
# Overlay RDF on single plot
fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(r_c, g_c, color="steelblue", lw=1.5, label="Crystalline")
ax.plot(r_a, g_a, color="orangered", lw=1.5, label="Amorphous")

ax.set_xlabel("r (Å)")
ax.set_ylabel("g(r)")
ax.set_title("Total RDF: crystalline vs amorphous SiO₂", fontweight="bold")
ax.set_xlim(0, 8)
ax.legend()
plt.tight_layout()
plt.savefig("rdf_overlay_SiO2.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved -> rdf_overlay_SiO2.png")

### Coordination number analysis

In crystalline SiO₂, Si is 4-coordinated by O (tetrahedral).
This tetrahedral motif is largely preserved in the amorphous phase.

In [ ]:
from ase.neighborlist import neighbor_list

def coordination_numbers(atoms, central, neighbour, cutoff):
    """Compute CN of 'central' atoms surrounded by 'neighbour' atoms."""
    i_arr, j_arr = neighbor_list('ij', atoms, cutoff)
    syms = np.array(atoms.get_chemical_symbols())
    central_idx = np.where(syms == central)[0]
    cn = np.zeros(len(central_idx), dtype=int)
    for k, ci in enumerate(central_idx):
        mask = (i_arr == ci) & (syms[j_arr] == neighbour)
        cn[k] = np.sum(mask)
    return cn

cutoff_SiO = 2.0  # Å — first coordination shell for Si-O

cn_crystal = coordination_numbers(crystal, "Si", "O", cutoff_SiO)
cn_amorphous = coordination_numbers(amorphous, "Si", "O", cutoff_SiO)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(cn_crystal, bins=range(0, 8), align="left",
         color="steelblue", edgecolor="white", alpha=0.8)
ax1.axvline(4, ls="--", color="coral", label="Tetrahedral CN=4")
ax1.set_xlabel("CN (Si–O)")
ax1.set_ylabel("Count")
ax1.set_title("Crystalline")
ax1.legend()
ax1.set_xlim(-0.5, 7.5)

ax2.hist(cn_amorphous, bins=range(0, 8), align="left",
         color="orangered", edgecolor="white", alpha=0.8)
ax2.axvline(4, ls="--", color="coral", label="Tetrahedral CN=4")
ax2.set_xlabel("CN (Si–O)")
ax2.set_ylabel("Count")
ax2.set_title("Amorphous")
ax2.legend()
ax2.set_xlim(-0.5, 7.5)

fig.suptitle(f"Si–O coordination number (cutoff {cutoff_SiO} Å)", fontsize=14)
plt.tight_layout()
plt.savefig("cn_comparison_SiO2.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Crystalline — mean CN: {np.mean(cn_crystal):.2f} ± {np.std(cn_crystal):.2f}")
print(f"Amorphous   — mean CN: {np.mean(cn_amorphous):.2f} ± {np.std(cn_amorphous):.2f}")
print(f"Expected    — CN = 4 (tetrahedral)")

### Bond angle distribution (O–Si–O)

In tetrahedral SiO₄, the ideal O–Si–O angle is 109.47°.
The amorphous phase should show a distribution centred near this value.

In [ ]:
def bond_angles(atoms, central, neighbour, cutoff):
    """Compute neighbour-central-neighbour angles."""
    i_arr, j_arr, d_arr, D_arr = neighbor_list('ijdD', atoms, cutoff)
    syms = np.array(atoms.get_chemical_symbols())
    central_idx = np.where(syms == central)[0]
    angles = []
    for ci in central_idx:
        mask = (i_arr == ci) & (syms[j_arr] == neighbour)
        vecs = D_arr[mask]
        n_nbr = len(vecs)
        for a in range(n_nbr):
            for b in range(a + 1, n_nbr):
                cos_angle = np.dot(vecs[a], vecs[b]) / (
                    np.linalg.norm(vecs[a]) * np.linalg.norm(vecs[b]))
                cos_angle = np.clip(cos_angle, -1, 1)
                angles.append(np.degrees(np.arccos(cos_angle)))
    return np.array(angles)

angles_crystal = bond_angles(crystal, "Si", "O", cutoff_SiO)
angles_amorphous = bond_angles(amorphous, "Si", "O", cutoff_SiO)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.hist(angles_crystal, bins=90, range=(0, 180),
         color="steelblue", edgecolor="none", alpha=0.7, density=True)
ax1.axvline(109.47, ls="--", color="coral", alpha=0.7, label="109.47° (tetrahedral)")
ax1.set_xlabel("O–Si–O angle (°)")
ax1.set_ylabel("Probability density")
ax1.set_title("Crystalline")
ax1.legend()

ax2.hist(angles_amorphous, bins=90, range=(0, 180),
         color="orangered", edgecolor="none", alpha=0.7, density=True)
ax2.axvline(109.47, ls="--", color="coral", alpha=0.7, label="109.47° (tetrahedral)")
ax2.set_xlabel("O–Si–O angle (°)")
ax2.set_ylabel("Probability density")
ax2.set_title("Amorphous")
ax2.legend()

fig.suptitle(f"O–Si–O bond angle distribution (cutoff {cutoff_SiO} Å)", fontsize=14)
plt.tight_layout()
plt.savefig("angles_SiO2.png", dpi=150, bbox_inches="tight")
plt.show()

### Summary

In [ ]:
print(f"{'Property':<30s}  {'Crystalline':>14s}  {'Amorphous':>14s}  {'Experimental':>14s}")
print("-" * 78)
print(f"{'Density (g/cm³)':<30s}  {rho_crystal:14.3f}  {rho_amorphous:14.3f}  {'~2.20':>14s}")
print(f"{'Mean Si-O CN':<30s}  {np.mean(cn_crystal):14.2f}  {np.mean(cn_amorphous):14.2f}  {'4.0':>14s}")
print(f"{'Mean O-Si-O angle (°)':<30s}  {np.mean(angles_crystal):14.1f}  {np.mean(angles_amorphous):14.1f}  {'109.47':>14s}")
print(f"\nNote: This is a quick tutorial run (~2000 K/ps cooling, CHGNet).")
print(f"For publication-quality structures, use MACE with 10-100 K/ps cooling")
print(f"rates and longer equilibration times (50-100 ps).")

---
## Note: Production-quality settings

The quick protocol above uses fast heating/cooling (~2000 K/ps) with CHGNet for
demonstration. For research-quality amorphous SiO₂ structures:

```python
pipe = MeltQuenchPipeline(
    input_file="SiO2_supercell.extxyz",
    work_dir="SiO2_production",
    cfg_override={
        "model": "mace-mpa-0",         # MACE for higher accuracy
        "device": "cuda",

        "opt": {"fmax": 0.01, "max_steps": 1000},

        "eq_premelt": {"ensemble": "NPT", "T": 300, "steps": 50000},       # 50 ps

        "melt": {
            "ensemble": "NPT", "T_start": 300, "T_end": 3000,
            "T_step": 100, "steps_per_T": 1000,                             # 100 K/ps
        },

        "eq_high": {"ensemble": "NPT", "T": 3000, "steps": 100000},        # 100 ps

        "quench": {
            "ensemble": "NPT", "T_start": 3000, "T_end": 300,
            "T_step": -100, "steps_per_T": 1000,                            # 100 K/ps
        },

        "eq_low": {"ensemble": "NPT", "T": 300, "steps": 50000},           # 50 ps
    },
)
```

Key differences from the tutorial protocol:
- **MACE** for higher accuracy (CHGNet is faster but less accurate for oxides)
- **1 fs timestep** (default) for better energy conservation
- **100 K/ps** cooling rate (20× slower)
- **50–100 ps** equilibration times (10–25× longer)
- **3000 K** melt temperature (higher for better disordering)

---
## Output summary

After running this tutorial, you should have:

```
SiO2_supercell.extxyz                    # 2x2x2 crystalline input
SiO2_melt_quench/
├── opt_stage.log                        # Stage 1 optimisation log
├── opt_stage.traj                       # Stage 1 trajectory
├── opt_final.cif                        # Stage 1 optimised crystal
├── stage2_eq.log                        # Stage 2 pre-melt eq log
├── stage2_eq.extxyz                     # Stage 2 trajectory + output
├── stage3_melt.log                      # Stage 3 melt log
├── stage3_melt.extxyz                   # Stage 3 trajectory
├── stage4_eq.log                        # Stage 4 high-T eq log
├── stage4_eq.extxyz                     # Stage 4 trajectory + output
├── stage5_quench.log                    # Stage 5 quench log
├── stage5_quench.extxyz                 # Stage 5 trajectory
├── stage6_eq.log                        # Stage 6 low-T eq log
├── stage6_eq.extxyz                     # Stage 6 trajectory + output
├── opt_stage.log                        # Stage 7 optimisation log
├── opt_final.cif                        # Stage 7 final amorphous structure
└── stage7_amorphous_final.cif           # Final output
rdf_comparison_SiO2.png
rdf_overlay_SiO2.png
cn_comparison_SiO2.png
angles_SiO2.png
```

Open `stage7_amorphous_final.cif` in VESTA to visualise the amorphous structure.